# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
metadata_obj = dataset.metadata
print(f"{getattr(metadata_obj, 'name', '')}: {getattr(metadata_obj, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their fields using their @id values
record_sets = dataset.metadata.record_sets

fields_per_record_set = {}
print("Available Record Sets and their Fields:")
for recset in record_sets:
    rs_id = getattr(recset, '@id', None)
    rs_name = getattr(recset, 'name', '')
    rs_fields = getattr(recset, 'fields', [])
    field_ids_and_names = []
    for f in rs_fields:
        field_id = getattr(f, '@id', None)
        field_name = getattr(f, 'name', '')
        field_ids_and_names.append((field_id, field_name))
    fields_per_record_set[rs_id] = field_ids_and_names
    print(f"  RecordSet @id: {rs_id}, Name: {rs_name}")
    for fid, fname in field_ids_and_names:
        print(f"    Field @id: {fid}, Name: {fname}")

# Show @id values for further referencing
all_record_set_ids = [getattr(r, '@id', None) for r in record_sets]
print("\nList of RecordSet @id values:")
for rsid in all_record_set_ids:
    print(f"  {rsid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract data for all record sets found.

dataframes = {}
for rec_set in dataset.metadata.record_sets:
    rec_set_id = getattr(rec_set, '@id', None)
    print(f"Loading records for RecordSet: {rec_set_id}")
    try:
        records_iter = dataset.records(record_set=rec_set_id)
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[rec_set_id] = df
        print(f"Columns for {rec_set_id}: {list(df.columns)}\n")
    except Exception as e:
        print(f"Could not load records for {rec_set_id}: {e}\n")

# Pick the first record set for continued analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Using {main_record_set_id} for demonstration.")
    print(dataframes[main_record_set_id].head())
else:
    print("No record sets found or could not load records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# If data is present, proceed with numeric field analysis
if dataframes:
    df = dataframes[main_record_set_id]
    # Infer likely numeric fields based on dtype or name
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to find numeric fields by name if inference failed
        for col in df.columns:
            if any(x in col.lower() for x in ['count', 'abundance', 'weight', 'coverage', 'score', 'mw']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if df[col].notnull().any():
                        numeric_candidates.append(col)
                except Exception as _: pass
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75)  # 75th percentile as example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field present in df
        group_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found for analysis.")
else:
    print("No dataframes available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    # Visualize the numeric field distribution before and after normalization
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.subplot(1, 2, 2)
    if f'{numeric_field}_normalized' in filtered_df:
        sns.histplot(filtered_df[f'{numeric_field}_normalized'].dropna(), kde=True)
        plt.title(f'Normalized {numeric_field} (filtered)')
    plt.tight_layout()
    plt.show()

    # If grouped statistics
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f'Average {numeric_field} by {group_field} (filtered)')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we demonstrated how to load and explore a Croissant-structured dataset using the `mlcroissant` library. We reviewed record set and field metadata (referenced by their `@id`s), extracted data into pandas DataFrames, performed basic filtering and normalization on numeric fields, grouped data for summary, and visualized resulting distributions. To adapt this notebook for deeper domain analysis, substitute the field and record set `@id`s with those appropriate to your application, and expand EDA and visualization as needed for your research questions.*